In [1]:
!pip install ultralytics -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 68.7 MB/s eta 0:00:00


In [2]:
import os
import copy
import time
import math
import random
import zipfile
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from PIL import Image
from ultralytics import YOLO

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("torch:", torch.__version__)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
torch: 2.11.0+cu128


In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True

set_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: NVIDIA A100-SXM4-40GB


In [5]:
AUTHOR_ROOT = "/content/drive/MyDrive/Image beam"
POSITION_ROOT = "/content/drive/MyDrive/scenario23_paper_style_sequence_split"
ZIP_PATH = "/content/drive/MyDrive/scenario23_dev_w_resources.zip"

FULL_CSV_PATH = os.path.join(AUTHOR_ROOT, "scenario23_img_beam.csv")

EXTRACT_ROOT = "/content/scenario23_dev_w_resources"
YOLO_CROP_ROOT = "/content/yolo_uav_crops_scenario23"

INDEX_COL = "index"
IMAGE_COL = "unit1_rgb"
LABEL_COL = "unit1_beam"

print("AUTHOR_ROOT:", os.path.exists(AUTHOR_ROOT), AUTHOR_ROOT)
print("POSITION_ROOT:", os.path.exists(POSITION_ROOT), POSITION_ROOT)
print("ZIP_PATH:", os.path.exists(ZIP_PATH), ZIP_PATH)
print("FULL_CSV_PATH:", os.path.exists(FULL_CSV_PATH), FULL_CSV_PATH)

assert os.path.exists(AUTHOR_ROOT)
assert os.path.exists(POSITION_ROOT)
assert os.path.exists(ZIP_PATH)
assert os.path.exists(FULL_CSV_PATH)

AUTHOR_ROOT: True /content/drive/MyDrive/Image beam
POSITION_ROOT: True /content/drive/MyDrive/scenario23_paper_style_sequence_split
ZIP_PATH: True /content/drive/MyDrive/scenario23_dev_w_resources.zip
FULL_CSV_PATH: True /content/drive/MyDrive/Image beam/scenario23_img_beam.csv


In [6]:
X_pos_train = np.load(os.path.join(POSITION_ROOT, "X_train_seq.npy"))
y_train = np.load(os.path.join(POSITION_ROOT, "y_train_seq.npy"))

X_pos_val = np.load(os.path.join(POSITION_ROOT, "X_val_seq.npy"))
y_val = np.load(os.path.join(POSITION_ROOT, "y_val_seq.npy"))

X_pos_test = np.load(os.path.join(POSITION_ROOT, "X_test_seq.npy"))
y_test = np.load(os.path.join(POSITION_ROOT, "y_test_seq.npy"))

print("Position sequences:")
print("train:", X_pos_train.shape, y_train.shape)
print("val  :", X_pos_val.shape, y_val.shape)
print("test :", X_pos_test.shape, y_test.shape)

assert X_pos_train.shape == (7968, 4, 21)
assert X_pos_val.shape == (2276, 4, 21)
assert X_pos_test.shape == (1139, 4, 21)

num_classes = int(max(y_train.max(), y_val.max(), y_test.max()) + 1)
pos_input_dim = X_pos_train.shape[-1]

print("num_classes:", num_classes)
print("pos_input_dim:", pos_input_dim)

Position sequences:
train: (7968, 4, 21) (7968,)
val  : (2276, 4, 21) (2276,)
test : (1139, 4, 21) (1139,)
num_classes: 29
pos_input_dim: 21


In [7]:
full_img_df = pd.read_csv(FULL_CSV_PATH)
full_img_df = full_img_df.sort_values(INDEX_COL).reset_index(drop=True)

print("Full image CSV:", full_img_df.shape)
print("Columns:", full_img_df.columns.tolist())

for col in [INDEX_COL, IMAGE_COL, LABEL_COL]:
    assert col in full_img_df.columns, f"Missing column: {col}"

display(full_img_df.head())
display(full_img_df.tail())

Full image CSV: (11387, 3)
Columns: ['index', 'unit1_rgb', 'unit1_beam']


,index,unit1_rgb,unit1_beam
0,1,../scenario23_dev/unit1/camera_data/image_BS1_...,22
1,2,../scenario23_dev/unit1/camera_data/image_BS1_...,22
2,3,../scenario23_dev/unit1/camera_data/image_BS1_...,22
3,4,../scenario23_dev/unit1/camera_data/image_BS1_...,22
4,5,../scenario23_dev/unit1/camera_data/image_BS1_...,20


,index,unit1_rgb,unit1_beam
11382,11383,../scenario23_dev/unit1/camera_data/image_BS1_...,20
11383,11384,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11384,11385,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11385,11386,../scenario23_dev/unit1/camera_data/image_BS1_...,19
11386,11387,../scenario23_dev/unit1/camera_data/image_BS1_...,19


In [8]:
os.makedirs(EXTRACT_ROOT, exist_ok=True)

marker_file = os.path.join(EXTRACT_ROOT, ".extracted_done")

if not os.path.exists(marker_file):
    print("Extracting scenario23 zip. Wait...")
    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        z.extractall(EXTRACT_ROOT)

    with open(marker_file, "w") as f:
        f.write("done")

    print("Extraction completed.")
else:
    print("Already extracted. Skipping.")

print("Extract root sample:")
print(os.listdir(EXTRACT_ROOT)[:30])

Extracting scenario23 zip. Wait...
Extraction completed.
Extract root sample:
['scenario23_dev', '.extracted_done']


In [9]:
image_files = []

for root, dirs, files in os.walk(EXTRACT_ROOT):
    for f in files:
        if f.lower().endswith((".jpg", ".jpeg", ".png", ".bmp")):
            image_files.append(os.path.join(root, f))

print("Total extracted image files:", len(image_files))
for p in image_files[:10]:
    print(p)

assert len(image_files) > 0, "No images found after extraction."

image_lookup_basename = {}
image_lookup_suffix = {}

for p in image_files:
    norm_p = p.replace("\\", "/")
    base = os.path.basename(norm_p)
    image_lookup_basename[base] = p

    parts = norm_p.split("/")
    for n in [2, 3, 4, 5, 6, 7, 8]:
        if len(parts) >= n:
            image_lookup_suffix["/".join(parts[-n:])] = p

def resolve_raw_image_path(path_from_csv):
    path_from_csv = str(path_from_csv).replace("\\", "/")
    clean = path_from_csv.lstrip("./").lstrip("/")

    if os.path.exists(path_from_csv):
        return path_from_csv

    candidate = os.path.join(EXTRACT_ROOT, clean)
    if os.path.exists(candidate):
        return candidate

    base = os.path.basename(clean)
    if base in image_lookup_basename:
        return image_lookup_basename[base]

    parts = clean.split("/")
    for n in [8, 7, 6, 5, 4, 3, 2]:
        if len(parts) >= n:
            suffix = "/".join(parts[-n:])
            if suffix in image_lookup_suffix:
                return image_lookup_suffix[suffix]

    return None

full_img_df["raw_image_path"] = full_img_df[IMAGE_COL].apply(resolve_raw_image_path)

missing = full_img_df["raw_image_path"].isna().sum()
print("Missing raw image paths:", missing, "of", len(full_img_df))

assert missing == 0

display(full_img_df[[INDEX_COL, IMAGE_COL, "raw_image_path", LABEL_COL]].head())

Total extracted image files: 11387
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_10994_17_58_52.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_11289_18_00_08.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_3683_17_08_47.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_6813_17_46_20.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_3738_17_08_55.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_6380_17_45_10.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_6420_17_45_16.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_10711_17_58_05.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_3609_17_08_35.jpg
/content/scenario23_dev_w_resources/scenario23_dev/unit1/camera_data/image_BS1_6585_17_45

,index,unit1_rgb,raw_image_path,unit1_beam
0,1,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
1,2,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
2,3,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
3,4,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,22
4,5,../scenario23_dev/unit1/camera_data/image_BS1_...,/content/scenario23_dev_w_resources/scenario23...,20


In [ ]:
yolo_model = YOLO("yolov8n.pt")

os.makedirs(YOLO_CROP_ROOT, exist_ok=True)

def yolo_crop_image(image_path, save_root=YOLO_CROP_ROOT, conf=0.10, padding=30):
    image_path = str(image_path)

    base = os.path.basename(image_path)
    save_path = os.path.join(save_root, base)

    if os.path.exists(save_path):
        return save_path

    img = Image.open(image_path).convert("RGB")
    w, h = img.size

    results = yolo_model.predict(source=image_path, conf=conf, verbose=False)
    boxes = results[0].boxes

    if boxes is None or len(boxes) == 0:
        img.save(save_path)
        return save_path

    xyxy = boxes.xyxy.cpu().numpy()
    areas = (xyxy[:, 2] - xyxy[:, 0]) * (xyxy[:, 3] - xyxy[:, 1])
    best_idx = areas.argmax()

    x1, y1, x2, y2 = xyxy[best_idx]

    x1 = max(int(x1) - padding, 0)
    y1 = max(int(y1) - padding, 0)
    x2 = min(int(x2) + padding, w)
    y2 = min(int(y2) + padding, h)

    crop = img.crop((x1, y1, x2, y2))

    if crop.size[0] < 10 or crop.size[1] < 10:
        img.save(save_path)
        return save_path

    crop.save(save_path)
    return save_path

sample_raw = full_img_df["raw_image_path"].iloc[0]
sample_crop = yolo_crop_image(sample_raw)

print("Sample raw:", sample_raw)
print("Sample crop:", sample_crop)

plt.figure(figsize=(5, 5))
plt.imshow(Image.open(sample_raw).convert("RGB"))
plt.axis("off")
plt.title("Raw Image")
plt.show()

plt.figure(figsize=(5, 5))
plt.imshow(Image.open(sample_crop).convert("RGB"))
plt.axis("off")
plt.title("YOLO Crop / Fallback")
plt.show()

In [ ]:
crop_paths = []

start = time.time()

for i, p in enumerate(full_img_df["raw_image_path"]):
    crop_paths.append(yolo_crop_image(p))

    if (i + 1) % 500 == 0:
        elapsed = (time.time() - start) / 60
        print(i + 1, "/", len(full_img_df), "| elapsed min:", round(elapsed, 2))

full_img_df["image_path"] = crop_paths

print("YOLO crop/fallback completed:", len(crop_paths))